# Spec Mode Comparison

Compares `example` and `specification` modes without calling an LLM. The script builds each mode's deterministic bundle for every sample task, runs TLC for `example` mode and TLAPS for `specification` mode, then reports pass rate, checker metric, and checker runtime. This measures template/checker overhead, not live LLM success rate.

In [ ]:
from pathlib import Path
import json, statistics, sys, time

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from agent.specs import deterministic_fallback_bundle, parse_spec_bundle
from agent.tlc import run_tlc
from agent.tlaps import run_tlaps


In [ ]:
tasks = json.loads((ROOT / 'tasks' / 'sample_tasks.json').read_text(encoding='utf-8'))
rows = []
RUNS = 5

for _ in range(RUNS):
    for task in tasks:
        for mode in ('example', 'specification'):
            raw = deterministic_fallback_bundle(task['signature'], task['public_tests'], mode)
            bundle = parse_spec_bundle(raw)
            start = time.perf_counter()
            if mode == 'specification':
                result = run_tlaps(bundle.module, bundle.tla)
                checker = 'TLAPS'
                metric = result.obligations
            else:
                result = run_tlc(bundle.module, bundle.tla, bundle.cfg)
                checker = 'TLC'
                metric = result.states_found
            rows.append({
                'task': task['name'],
                'mode': mode,
                'checker': checker,
                'passed': result.passed,
                'metric': metric,
                'seconds': time.perf_counter() - start,
                'structured': bool(bundle.structured_spec),
            })


In [ ]:
print('| task | mode | checker | pass/runs | mean states/obligations | mean checker seconds | structured spec |')
print('|---|---:|---:|---:|---:|---:|---:|')
for task in tasks:
    for mode in ('example', 'specification'):
        subset = [r for r in rows if r['task'] == task['name'] and r['mode'] == mode]
        passed = sum(1 for r in subset if r['passed'])
        metrics = [r['metric'] for r in subset if r['metric'] is not None]
        seconds = [r['seconds'] for r in subset]
        structured = any(r['structured'] for r in subset)
        print(f"| {task['name']} | {mode} | {subset[0]['checker']} | {passed}/{len(subset)} | {statistics.mean(metrics):.1f} | {statistics.mean(seconds):.3f} | {'yes' if structured else 'no'} |")


For live provider-backed comparison, run `python tests/eval.py --compare-spec-modes` from the repository root after configuring `OPENAI_API_KEY` or `GEMINI_API_KEY`.